In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.utils

>DOM utils 
output-file: core.dom.utils.html
title: core.dom.utils

This notebook provides utility functions for DOM manipulation.
---

In [ ]:
# | default_exp core.dom.utils

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

InteractiveShell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.basics import patch
from fastcore.test import *


In [ ]:
# | export
import asyncio
import os
import re
from pathlib import Path

import markdown


In [ ]:
#| export
from ollama import AsyncClient, ChatResponse, chat
try:
    from openai import AsyncOpenAI
except ModuleNotFoundError:
    class AsyncOpenAI:
        def __init__(self, *args, **kwargs):
            pass

from collections.abc import Callable
from pydantic import BaseModel, Field
from enum import Enum
from functools import wraps
from inspect import Parameter, signature

import jsoncfg
from jsoncfg.config_classes import (
    ConfigJSONArray,
    ConfigJSONObject,
    ConfigJSONScalar,
    ConfigNode,
)

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide
import base64
import threading
from tempfile import TemporaryDirectory
from types import SimpleNamespace
from unittest.mock import patch as mock_patch

def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result = {}
    error = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')

class FakeChatResponse:
    def __init__(self, content):
        self.message = SimpleNamespace(content=content)

class FakeAsyncChatClient:
    def __init__(self, content='ok'):
        self.content = content
        self.calls = []

    async def chat(self, **kwargs):
        self.calls.append(kwargs)
        return FakeChatResponse(self.content)

class FakeOpenAIClient:
    def __init__(self, content='ok'):
        self.content = content
        self.calls = []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    async def _create(self, **kwargs):
        self.calls.append(kwargs)
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=self.content))])

def write_test_image(root: Path, name: str = 'robot.png') -> Path:
    path = root / name
    path.write_bytes(base64.b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO+a8GkAAAAASUVORK5CYII='))
    return path

In [ ]:
# | export

# OpenAI client for text summarization 
openai_client = AsyncOpenAI(
    api_key=os.environ.get('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# Ollama client for embeddings and image analysis
ollama_client = AsyncClient()

# For backward compatibility, keep chat_client pointing to OpenAI client
chat_client = openai_client

In [ ]:
import importlib.util
import os
import shutil
import sys

print(f"sys.executable: {sys.executable}")
print(f"VIRTUAL_ENV: {os.environ.get('VIRTUAL_ENV')}")
print(f"which python: {shutil.which('python')}")
ollama_spec = importlib.util.find_spec('ollama')
print(f"ollama importable: {ollama_spec is not None}")
if ollama_spec is not None:
    import ollama
    print(f"ollama module: {getattr(ollama, '__file__', 'unknown')}")

In [ ]:
#| export


model = "deepseek-v4-flash:cloud"  # "deepseek-r1:32b" "gemma3:27b"
if not ("cloud" in model):
    chat_client: AsyncClient = AsyncClient()   
else:
    chat_client = AsyncClient(
        host="http://ollama.com",
        headers={"Authorization": f"Bearer {os.environ.get('OLLAMA_API_KEY')}"}
        )
msg_queue: asyncio.Queue = asyncio.Queue(maxsize=8)  # Limit the queue size to 8 messages
resp_queue: asyncio.Queue = asyncio.Queue(maxsize=1)  # Limit the queue size to 1 message for response 
lock = asyncio.Lock()  # Lock to ensure thread-safe access to the queues

In [ ]:
#| export


model = "qwen-plus-2025-07-28"  #  "deepseek-v3.2"  #  
chat_client = AsyncOpenAI(
    api_key=os.environ.get('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )

In [ ]:
#| export
ResponseStatus = Enum("ResponseStatus", ["PENDING", "PROCESSING", "REORGNIZED", "SEMANTICIZED", "CANCELLED", "COMPLETED", "ERROR"])
class AnalysisStatus(BaseModel):
    """
    Enum to represent the status of the analysis process.
    """
    status: ResponseStatus = Field(ResponseStatus.PENDING, description="Current status of the analysis process")
    exception: str = Field("", description="Exception message if any error occurs during analysis")

In [ ]:
#| export
def _config_error(message: str, config_node: ConfigNode, node=None) -> ValueError:
    line = jsoncfg.node_location(config_node).line if config_node is not None else "unknown"
    details = [message, f"On line: {line}"]
    if node is not None:
        details.insert(1, f"Node: {node}")
    return ValueError("\n".join(details))

In [ ]:
# | hide
def test_config_error_reports_message_line_and_node():
    config = jsoncfg.loads_config('{"item": 1}')['item']
    err = _config_error('boom', config, {'kind': 'value'})
    test_eq('boom' in str(err), True)
    test_eq('Node:' in str(err), True)
    test_eq('kind' in str(err), True)
    test_eq('value' in str(err), True)
    test_eq('On line:' in str(err), True)
    test_eq('unknown' in str(_config_error('boom', None)), True)


test_config_error_reports_message_line_and_node()

In [ ]:
#| export
def with_async_context(context_builder: Callable[..., str]) -> Callable:
    """Wrap async helpers to append contextual information to unexpected errors."""
    def build_context(*args, **kwargs) -> str:
        try:
            return context_builder(*args, **kwargs)
        except TypeError:
            params = tuple(signature(context_builder).parameters.values())
            if any(param.kind is Parameter.VAR_POSITIONAL for param in params):
                context_args = args
            else:
                positional_limit = sum(
                    param.kind in (Parameter.POSITIONAL_ONLY, Parameter.POSITIONAL_OR_KEYWORD)
                    for param in params
                )
                context_args = args[:positional_limit]

            if any(param.kind is Parameter.VAR_KEYWORD for param in params):
                context_kwargs = kwargs
            else:
                allowed_kwargs = {
                    param.name
                    for param in params
                    if param.kind in (Parameter.KEYWORD_ONLY, Parameter.POSITIONAL_OR_KEYWORD)
                }
                context_kwargs = {key: value for key, value in kwargs.items() if key in allowed_kwargs}

            return context_builder(*context_args, **context_kwargs)

    def decorator(func: Callable) -> Callable:
        @wraps(func)
        async def wrapper(*args, **kwargs):
            try:
                return await func(*args, **kwargs)
            except ValueError:
                raise
            except Exception as exc:
                raise ValueError(f"{build_context(*args, **kwargs)}\nError: {exc}") from exc
        return wrapper
    return decorator

In [ ]:
# | hide
def test_with_async_context_wraps_runtime_errors_and_preserves_value_errors():
    @with_async_context(lambda config_node, node: f'ctx:{node}')
    async def wrapped(config_node, node, extra=None):
        raise RuntimeError('broken')

    @with_async_context(lambda config_node, node: f'ctx:{node}')
    async def wrapped_value_error(config_node, node):
        raise ValueError('original')

    test_fail(lambda: run_async(wrapped('cfg', {'a': 1}, extra='x')), contains='ctx:{')
    test_fail(lambda: run_async(wrapped_value_error('cfg', {'a': 1})), contains='original')


test_with_async_context_wraps_runtime_errors_and_preserves_value_errors()

In [ ]:
# | export
def get_text_summary_response(content: str, model:str="gemma3-27b", role:str="user", lang: str="zh") -> ChatResponse:
    """Generate a text-summary chat response for the given content.
    
    Args:
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    match lang:
        case "en":
            prompt = (
                f"Please provide a summary of the following string. "
                f"The summary should be concise and informative: {content}. "
            )
        case "zh":
            content = re.sub(r"\s+", " ", content.strip())
            prompt = (
                f"请提供以下字符串的摘要。"
                f"摘要应简明扼要且信息丰富: {content}. "
            )
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    return chat(
        model=model,
        messages=[
            {
                "role": role,
                "content": prompt,
            }
        ],
    )

In [ ]:
# | hide
def test_get_text_summary_response_builds_prompt_and_validates_lang():
    captured = {}
    def fake_chat(model, messages):
        captured['model'] = model
        captured['messages'] = messages
        return FakeChatResponse('done')
    with mock_patch.dict(globals(), {'chat': fake_chat}):
        response = get_text_summary_response('  hello\nworld  ', model='m', role='user', lang='zh')
    test_eq(response.message.content, 'done')
    test_eq('hello world' in captured['messages'][0]['content'], True)
    test_fail(lambda: get_text_summary_response('x', lang='jp'), contains='Unsupported language')


test_get_text_summary_response_builds_prompt_and_validates_lang()

In [ ]:
# | export
image_link_pattern = r'[^\s]+\.(?:jpg|jpeg|png|gif|bmp|webp)'

In [ ]:
# | export
def get_image_summary_response(image_link: str | Path, model:str="gemma3:27b", role:str="user", lang: str='zh') -> ChatResponse:
    """Generate an image-summary chat response for a local image path.
    
    Args:
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    if isinstance(image_link, Path):
        image_link = str(image_link)
    if not re.match(image_link_pattern, image_link):
        # If the image link is not a URL, throw an error
        raise ValueError(f"Invalid image link: {image_link}")
    
    match lang:
        case 'en':
            prompt = "Please provide a summary of the following image. The summary should be concise and informative about the robot."
        case 'zh':
            prompt = "请提供以下图像的摘要。关于机器人机械尺寸,运动范围,自由度的说明应简明扼要。"
        case _:
            raise ValueError(f"Unsupported language: {lang}")

    response = chat(
        model=model,
        messages=[
            {
                "role": role,
                "content": prompt,
                'images': [f"{image_link}"],
            }
        ],
    )
    return response


In [ ]:
# | hide
def test_get_image_summary_response_validates_link_and_lang():
    captured = {}
    def fake_chat(model, messages):
        captured['messages'] = messages
        return FakeChatResponse('img')
    with mock_patch.dict(globals(), {'chat': fake_chat}):
        response = get_image_summary_response('img/robot.png', model='m', role='user', lang='en')
    test_eq(response.message.content, 'img')
    test_eq(captured['messages'][0]['images'], ['img/robot.png'])
    test_fail(lambda: get_image_summary_response('img/robot.txt'), contains='Invalid image link')
    test_fail(lambda: get_image_summary_response('img/robot.png', lang='jp'), contains='Unsupported language')


test_get_image_summary_response_validates_link_and_lang()

In [ ]:
# | export
async def get_text_summary_response_async(client: AsyncClient, content: str, model:str="gemma3-27b", role:str="user", lang: str="zh") -> ChatResponse:
    """Generate a text-summary chat response with an async Ollama client.
    
    Args:
        client: Async Ollama client used to send the request.
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    content = re.sub(r"\s+", " ", content.strip())
    match lang:
        case "en":
            prompt = (
                f"Please provide a summary of the following string. "
                f"The summary should be concise and informative: {content}. "
            )
        case "zh":
            prompt = (
                f"请提供以下字符串的摘要。"
                f"摘要应简明扼要且信息丰富: {content}. "
            )
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    
    message = {
        "role": role,
        "content": prompt,
    }
    response =  await client.chat(
        model=model,
        messages=[message],
    )
    return response

In [ ]:
# | hide
def test_get_text_summary_response_async_uses_client_chat():
    client = FakeAsyncChatClient('async-text')
    response = run_async(get_text_summary_response_async(client, '  hello\nworld  ', model='m', role='user', lang='zh'))
    test_eq(response.message.content, 'async-text')
    test_eq('hello world' in client.calls[0]['messages'][0]['content'], True)
    test_fail(lambda: run_async(get_text_summary_response_async(client, 'x', lang='jp')), contains='Unsupported language')


test_get_text_summary_response_async_uses_client_chat()

In [ ]:
# | export
async def get_image_summary_response_async(client: AsyncClient, image_link: str | Path, model:str="gemma3:27b", role:str="user", lang: str='zh') -> ChatResponse:
    """Generate an image-summary chat response with an async Ollama client.
    
    Args:
        client: Async Ollama client used to send the request.
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    if isinstance(image_link, Path):
        image_link = str(image_link)
    # if not re.match(image_link_pattern, image_link):
    #     # If the image link is not a URL, throw an error
    #     raise ValueError(f"Invalid image link: {image_link}")
    
    match lang:
        case 'en':
            prompt = "Please provide a summary of the following image. The summary should be concise and informative about the robot."
        case 'zh':
            prompt = "请提供以下图像的摘要。关于机器人机械尺寸,运动范围,自由度的说明应简明扼要。"
        case _:
            raise ValueError(f"Unsupported language: {lang}")

    response = await client.chat(
        model=model,
        messages=[
            {
                "role": role,
                "content": prompt,
                'images': [f"{image_link}"],
            }
        ],
    )
    return response


In [ ]:
# | hide
def test_get_image_summary_response_async_uses_client_chat():
    client = FakeAsyncChatClient('async-image')
    response = run_async(get_image_summary_response_async(client, Path('img/robot.png'), model='m', role='user', lang='en'))
    test_eq(response.message.content, 'async-image')
    test_eq(client.calls[0]['messages'][0]['images'], ['img/robot.png'])
    test_fail(lambda: run_async(get_image_summary_response_async(client, 'img/robot.png', lang='jp')), contains='Unsupported language')


test_get_image_summary_response_async_uses_client_chat()

In [ ]:
# | export
async def get_text_summary_response_openai_async(client: AsyncOpenAI, content: str, model:str="qwen-plus-2025-07-28", role:str="user", lang: str="zh") -> str:
    """Generate a text summary with an async OpenAI-compatible client.
    
    Args:
        client: Async OpenAI-compatible client used to send the request.
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    match lang:
        case "en":
            prompt = (
                f"Please provide a summary of the following string. "
                f"The summary should be concise and informative: {content}. "
            )
        case "zh":
            content = re.sub(r"\s+", " ", content.strip())
            prompt = (
                f"请提供以下字符串的摘要。"
                f"摘要应简明扼要且信息丰富: {content}. "
            )
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": role, "content": prompt}],
        extra_headers={"X-RateLimit-RPM": "60", "X-RateLimit-TPM": "1000000"}
    )
    
    return response.choices[0].message.content or ""

In [ ]:
# | hide
def test_get_text_summary_response_openai_async_uses_completions_api():
    client = FakeOpenAIClient('openai-text')
    response = run_async(get_text_summary_response_openai_async(client, '  hello\nworld  ', model='m', role='user', lang='zh'))
    test_eq(response, 'openai-text')
    test_eq('hello world' in client.calls[0]['messages'][0]['content'], True)
    test_fail(lambda: run_async(get_text_summary_response_openai_async(client, 'x', lang='jp')), contains='Unsupported language')


test_get_text_summary_response_openai_async_uses_completions_api()

In [ ]:
# | export
async def get_image_summary_response_openai_async(client: AsyncOpenAI, image_link: str | Path, model:str="qwen-vl-plus-2025-01-26", role:str="user", lang: str='zh') -> str:
    """Generate an image summary with an async OpenAI-compatible client.
    
    Args:
        client: Async OpenAI-compatible client used to send the request.
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    if isinstance(image_link, Path):
        image_link = str(image_link)
    if not re.search(image_link_pattern, image_link):
        raise ValueError(f"Invalid image link: {image_link}")
    
    # Read and encode image as base64
    import base64
    with open(image_link, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
    match lang:
        case 'en':
            prompt = "Please provide a summary of the following image. The summary should be concise and informative about the robot."
        case 'zh':
            prompt = "请提供以下图像的摘要。摘要应简明扼要，信息丰富，描述所显示的机器人内容。"
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    
    response = await client.chat.completions.create(
        model=model,
        messages=[{
            "role": role,
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]
        }], # type: ignore
        extra_headers={"X-RateLimit-RPM": "60", "X-RateLimit-TPM": "1000000"}
    )
    
    return response.choices[0].message.content or ""

In [ ]:
# | hide
def test_get_image_summary_response_openai_async_validates_and_encodes_image():
    client = FakeOpenAIClient('openai-image')
    with TemporaryDirectory() as tmpdir:
        image_path = write_test_image(Path(tmpdir))
        response = run_async(get_image_summary_response_openai_async(client, image_path, model='m', role='user', lang='zh'))
        test_fail(lambda: run_async(get_image_summary_response_openai_async(client, image_path, lang='jp')), contains='Unsupported language')
    test_eq(response, 'openai-image')
    payload = client.calls[0]['messages'][0]['content'][1]['image_url']['url']
    test_eq(payload.startswith('data:image/jpeg;base64,'), True)
    test_fail(lambda: run_async(get_image_summary_response_openai_async(client, 'img/robot.txt')), contains='Invalid image link')


test_get_image_summary_response_openai_async_validates_and_encodes_image()

In [ ]:
# | export
async def get_summary_response_async(client: AsyncClient | AsyncOpenAI, content: str, model:str="gemma3:27b", role:str="user", lang: str="zh") -> str:
    """Route text summarization to the matching async client implementation.
    
    Args:
        client: Async Ollama or OpenAI-compatible client.
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    if isinstance(client, AsyncClient):
        # Ollama client
        response = await get_text_summary_response_async(client, content, model, role, lang)
        return response.message.content if hasattr(response, 'message') and hasattr(response.message, 'content') else "" # type: ignore
    elif isinstance(client, AsyncOpenAI):
        # OpenAI client
        return await get_text_summary_response_openai_async(client, content, model, role, lang)
    else:
        raise ValueError(f"Unsupported client type: {type(client)}")  # noqa: TRY004


In [ ]:
# | hide
def test_get_summary_response_async_routes_by_client_type():
    async def fake_ollama(client, content, model, role, lang):
        return FakeChatResponse('ollama')
    async def fake_openai(client, content, model, role, lang):
        return 'openai'
    openai_like = AsyncOpenAI(api_key='test', base_url='https://example.invalid/v1')
    with mock_patch.dict(globals(), {'get_text_summary_response_async': fake_ollama, 'get_text_summary_response_openai_async': fake_openai}):
        test_eq(run_async(get_summary_response_async(AsyncClient(), 'x')), 'ollama')
        test_eq(run_async(get_summary_response_async(openai_like, 'x')), 'openai')
    test_fail(lambda: run_async(get_summary_response_async(object(), 'x')), contains='Unsupported client type')


test_get_summary_response_async_routes_by_client_type()

In [ ]:
# | export
async def get_image_summary_async(client: AsyncClient | AsyncOpenAI, image_link: str | Path, model:str="gemma3:27b", role:str="user", lang: str='zh') -> str:
    """Route image summarization to the matching async client implementation.
    
    Args:
        client: Async Ollama or OpenAI-compatible client.
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    if isinstance(client, AsyncClient):
        # Ollama client
        response = await get_image_summary_response_async(client, image_link, model, role, lang)
        return response.message.content if hasattr(response, 'message') and hasattr(response.message, 'content') else "" # type: ignore
    elif isinstance(client, AsyncOpenAI):
        # OpenAI client  
        return await get_image_summary_response_openai_async(client, image_link, model, role, lang)
    else:
        raise ValueError(f"Unsupported client type: {type(client)}")  # noqa: TRY004


In [ ]:
# | hide
def test_get_image_summary_async_routes_by_client_type():
    async def fake_ollama(client, image_link, model, role, lang):
        return FakeChatResponse('ollama-image')
    async def fake_openai(client, image_link, model, role, lang):
        return 'openai-image'
    openai_like = AsyncOpenAI(api_key='test', base_url='https://example.invalid/v1')
    with mock_patch.dict(globals(), {'get_image_summary_response_async': fake_ollama, 'get_image_summary_response_openai_async': fake_openai}):
        test_eq(run_async(get_image_summary_async(AsyncClient(), 'img/robot.png')), 'ollama-image')
        test_eq(run_async(get_image_summary_async(openai_like, 'img/robot.png')), 'openai-image')
    test_fail(lambda: run_async(get_image_summary_async(object(), 'img/robot.png')), contains='Unsupported client type')


test_get_image_summary_async_routes_by_client_type()

In [ ]:
# | hide
# Example exploration cell intentionally disabled during automated runs.
pass

In [ ]:
# | hide
# Example display cell intentionally disabled during automated runs.
pass

In [ ]:
# | hide
# Example regex exploration cell intentionally disabled during automated runs.
pass

In [ ]:
# | hide
# Live model demo intentionally disabled during automated runs.
pass

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()